In [ ]:
# Importar bibliotecas y configuración
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance

try:
    import xgboost as xgb
except Exception:
    xgb = None

try:
    import lightgbm as lgb
except Exception:
    lgb = None

try:
    from imblearn.over_sampling import SMOTE
except Exception:
    SMOTE = None

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_PATH = Path('/Users/cristobal/Desktop/Capstone/neuroflex_dataset_augmented.csv')
OUTPUT_DIR = Path('/Users/cristobal/Desktop/Capstone')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Configuración inicial lista.')


In [ ]:
# Cargar y explorar el dataset

df = pd.read_csv(DATA_PATH)
print(f'Forma del dataset: {df.shape}')
print('\nColumnas disponibles:')
print(df.columns.tolist())
print('\nVista previa:')
df.head()


In [ ]:
# Limpiar datos y manejar valores faltantes
# Asegurar tipos numéricos útiles
if 'age' in df.columns:
    df['age'] = pd.to_numeric(df['age'], errors='coerce').fillna(0).round().astype(int)

# Asegurar Precision como numérica y recortar valores fuera del rango clínico
if 'Precision' in df.columns:
    df['Precision'] = pd.to_numeric(df['Precision'], errors='coerce').clip(lower=0, upper=100)

# Verificar duplicados y nulos
print('Duplicados:', df.duplicated().sum())
print('\nValores nulos por columna:')
print(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False))

# Mostrar resumen de tipos
print('\nTipos de columnas:')
df.dtypes


In [ ]:
# Analizar distribuciones y la variable objetivo
# Definir una variable objetivo clínica más robusta y equilibrada
# Estrategia: separar rendimiento bajo versus no bajo
# bajo -> Precision < 35
# no bajo -> Precision >= 35

if 'Precision' in df.columns:
    df['performance_band'] = pd.cut(
        df['Precision'],
        bins=[-1, 35, 101],
        labels=['bajo', 'no_bajo'],
        right=False,
        include_lowest=True,
    )
    df['target'] = df['performance_band'].map({'bajo': 0, 'no_bajo': 1})

print('Distribución de la variable objetivo:')
print(df['target'].value_counts(normalize=True).sort_index())

# Ver distribución de Precision
plt.figure(figsize=(10, 5))
plt.hist(df['Precision'], bins=30, edgecolor='black')
plt.title('Distribución de Precision')
plt.xlabel('Precision')
plt.ylabel('Frecuencia')
plt.show()

# Boxplot por clase objetivo
if 'target' in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x='target', y='Precision')
    plt.title('Precision por clase objetivo')
    plt.xlabel('target')
    plt.ylabel('Precision')
    plt.show()


In [ ]:
# Visualizar correlaciones y relaciones de variables
# Matriz de correlación para variables numéricas
num_cols = df.select_dtypes(include=['number']).columns.tolist()

if len(num_cols) > 1:
    corr = df[num_cols].corr(numeric_only=True)
    plt.figure(figsize=(14, 10))
    sns.heatmap(corr, cmap='coolwarm', annot=False, fmt='.2f')
    plt.title('Matriz de correlación')
    plt.show()

# Histogramas de variables clave
key_features = [
    'age', 'TiempoRespuestaPararse', 'TiempoActivoTarea', 'CantAciertasTotales',
    'TiempoTutorial', 'TiempoRespuestaPregunta1', 'TiempoRespuestaPregunta2'
]
key_features = [c for c in key_features if c in df.columns]

if key_features:
    df[key_features].hist(figsize=(16, 12), bins=20)
    plt.tight_layout()
    plt.show()


In [ ]:
# Codificar variables categóricas
# Selección de columnas categóricas y numéricas
categorical_cols = [c for c in df.columns if df[c].dtype == 'object' and c != 'performance_band']
numeric_cols = [c for c in df.columns if c not in categorical_cols and c not in ['target', 'performance_band'] and pd.api.types.is_numeric_dtype(df[c])]

print('Categóricas:', categorical_cols)
print('Numéricas:', numeric_cols)

# Crear una copia del dataset con columnas útiles para modelado
model_df = df.copy()

# Eliminar columnas irrelevantes para el target, pero mantener Precision
# porque se usa para feature engineering y para definir la variable objetivo
exclude_cols = ['Performance', 'rendimiento', 'performance_band']
model_df = model_df.drop(columns=[c for c in exclude_cols if c in model_df.columns], errors='ignore')

# Un esquema de preprocesamiento robusto
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
    ]
)

preprocessor


In [ ]:
# Generar nuevas características (feature engineering)
# Crear indicadores clínicos útiles y bins para incrementar la señal predictiva
model_df['Precision_bin'] = pd.cut(model_df['Precision'], bins=[0, 25, 50, 75, 100], labels=['muy_baja', 'baja', 'media', 'alta'], include_lowest=True)
model_df['age_group'] = pd.cut(model_df['age'], bins=[0, 40, 60, 80, 120], labels=['joven', 'adulto', 'mayor', 'anciano'], include_lowest=True)
model_df['tiempo_total'] = model_df.get('TiempoRespuestaPararse', 0) + model_df.get('TiempoActivoTarea', 0) + model_df.get('TiempoTutorial', 0)
model_df['ratio_aciertos'] = model_df.get('CantAciertasTotales', 0) / (model_df.get('ObjetosInteractuadosCorrectamente', 0) + 1)

# Si existen columnas de tiempos, transformar a log para suavizar outliers
for col in ['TiempoRespuestaPararse', 'TiempoActivoTarea', 'TiempoTutorial', 'TiempoRespuestaPregunta1', 'TiempoRespuestaPregunta2', 'TiempoRespuestaPregunta3']:
    if col in model_df.columns:
        model_df[f'{col}_log'] = np.log1p(model_df[col].clip(lower=0))

# Mantener objetivo y columnas útiles
feature_cols = [c for c in model_df.columns if c not in ['target', 'performance_band', 'Precision']]
print('Número de features finales:', len(feature_cols))
feature_cols[:15]


In [ ]:
# Seleccionar variables relevantes
# Usar mutual information para priorizar variables
X = model_df[feature_cols]
y = model_df['target']

numeric_for_mi = X.select_dtypes(include=['number'])
if not numeric_for_mi.empty:
    mi = mutual_info_classif(numeric_for_mi, y, random_state=SEED)
    mi_df = pd.DataFrame({'feature': numeric_for_mi.columns, 'mi_score': mi}).sort_values('mi_score', ascending=False)
    print('\nTop features por mutual information:')
    print(mi_df.head(15))

# Por simplicidad, conservar todas las features relevantes y descartar columnas de metadatos
# (ID, fechas, UUID, etc.) si se encuentran
metadata_like = [c for c in feature_cols if 'date' in c.lower() or 'uuid' in c.lower() or 'id' in c.lower()]
feature_cols = [c for c in feature_cols if c not in metadata_like]
print('\nFeatures finaless para entrenamiento:', len(feature_cols))
feature_cols[:20]


In [ ]:
# Dividir datos en train, validation y test
X_final = model_df[feature_cols]
y_final = model_df['target']

X_train, X_temp, y_train, y_temp = train_test_split(
    X_final, y_final,
    test_size=0.30,
    random_state=SEED,
    stratify=y_final,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print('Train size:', X_train.shape)
print('Validation size:', X_val.shape)
print('Test size:', X_test.shape)
print('\nDistribución de clases (train):')
print(pd.Series(y_train).value_counts().sort_index())
print('\nDistribución de clases (validation):')
print(pd.Series(y_val).value_counts().sort_index())
print('\nDistribución de clases (test):')
print(pd.Series(y_test).value_counts().sort_index())


In [ ]:
# Escalar y transformar features
# Definir tipos de columnas para el preprocesador
categorical_model_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_model_cols = [c for c in X_train.columns if c not in categorical_model_cols]

preprocessor_model = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_model_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_model_cols),
    ]
)

preprocessor_model


# ¿Por qué `SVC` es un modelo de clasificación?

`SVC` significa `Support Vector Classifier`.

- `Classifier` indica que su salida es una clase, no un número continuo.
- En sklearn, `SVC` toma features de entrada y devuelve etiquetas como `0`, `1`, `bajo`, `alto`, etc.
- Internamente, busca un hiperplano que maximiza la separación entre clases.
- Por eso se usa para problemas de clasificación, como detectar si una persona está en una clase de rendimiento o no.

Es importante distinguirlo de `SVR`:
- `SVC` = clasificación
- `SVR` = regresión

En otras palabras, si la variable objetivo es discreta (clases), `SVC` es el modelo correcto. Si la variable objetivo es continua (precio, temperatura, tiempo), entonces sería `SVR` o otro modelo de regresión.

En este notebook, `SVC` se usa porque la variable objetivo es categórica (por ejemplo: `bajo` vs `no_bajo`).

In [ ]:
# Entrenar modelos base

models = {
    'LogisticRegression': LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED),
    'RandomForest': RandomForestClassifier(n_estimators=500, random_state=SEED, class_weight='balanced'),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=500, random_state=SEED, class_weight='balanced'),
    'GradientBoosting': GradientBoostingClassifier(random_state=SEED),
    'SVC': SVC(class_weight='balanced', random_state=SEED),
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor_model),
        ('model', model),
    ])
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)
    val_acc = accuracy_score(y_val, val_pred)
    val_prec = precision_score(y_val, val_pred, zero_division=0)
    val_rec = recall_score(y_val, val_pred, zero_division=0)
    val_f1 = f1_score(y_val, val_pred, zero_division=0)
    results.append({
        'model': name,
        'accuracy': val_acc,
        'precision': val_prec,
        'recall': val_rec,
        'f1': val_f1,
    })
    print(f'\n=== {name} ===')
    print(f'Accuracy: {val_acc:.4f}')
    print(f'Precision: {val_prec:.4f}')
    print(f'Recall: {val_rec:.4f}')
    print(f'F1: {val_f1:.4f}')
    print('Confusion matrix:\n', confusion_matrix(y_val, val_pred))

results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
print('\nRanking por F1 en validación:')
print(results_df)


In [ ]:
# Evaluar métricas iniciales
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    print('Accuracy:', round(acc, 4))
    print('Precision:', round(precision, 4))
    print('Recall:', round(recall, 4))
    print('F1:', round(f1, 4))
    print('Confusion matrix:\n', cm)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

# Usar el mejor modelo de validación para evaluar en test
best_model_name = results_df.iloc[0]['model']
best_model = models[best_model_name]

best_pipe = Pipeline([
    ('preprocessor', preprocessor_model),
    ('model', best_model),
])

print(f'\nModelo seleccionado para evaluación final: {best_model_name}')
final_metrics = evaluate_model(best_pipe, X_train, y_train, X_test, y_test)
final_metrics